### Import libraries

In [2]:
import pandas as pd
import numpy as np

### 2.Load the raw dataset

In [3]:
df = pd.read_csv("../data/raw/customer_contacts_dirty.csv")

df.head()

,customer_id,first_name,last_name,email,phone,company,city,country,signup_date,status
0,C0001,John,Smith,john.smith@example.com,-1070,Acme Inc,New York,USA,15-01-2024,Active
1,C0002,Sarah,Johnson,sarah.j@example.com,4155550102,BlueSky Ltd,Los Angeles,USA,01/22/2024,active
2,C0003,Michael,Brown,michael.brown@example.com,+1 415 555 0103,TechNova,Chicago,USA,03-02-2024,ACTIVE
3,C0004,Emily,Davis,emily.davis@example.com,NaN,GreenLeaf,Seattle,USA,12-02-2024,Inactive
4,C0005,David,Wilson,david.wilson@example.com,555-0105,CloudNine,Austin,USA,01-03-2024,Active


### 3.Understand the dataset

In [4]:
print("Shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nDataset information:")
df.info()

Shape: (150, 10)

Column names:
['customer_id', 'first_name', 'last_name', 'email', 'phone', 'company', 'city', 'country', 'signup_date', 'status']

Data types:
customer_id    str
first_name     str
last_name      str
email          str
phone          str
company        str
city           str
country        str
signup_date    str
status         str
dtype: object

Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    str  
 1   first_name   150 non-null    str  
 2   last_name    150 non-null    str  
 3   email        149 non-null    str  
 4   phone        147 non-null    str  
 5   company      150 non-null    str  
 6   city         150 non-null    str  
 7   country      150 non-null    str  
 8   signup_date  150 non-null    str  
 9   status       150 non-null    str  
dtypes: str(10)
memory usage: 11.8 KB


### 4.Inspect missing values

In [5]:
missing_values = df.isnull().sum()

print(missing_values)

customer_id    0
first_name     0
last_name      0
email          1
phone          3
company        0
city           0
country        0
signup_date    0
status         0
dtype: int64


In [6]:
missing_percentage = (
    df.isnull().sum() / len(df) * 100
).round(2)

missing_report = pd.DataFrame({
    "missing_count": missing_values,
    "missing_percentage": missing_percentage
})

missing_report

,missing_count,missing_percentage
customer_id,0,0.00
first_name,0,0.00
last_name,0,0.00
email,1,0.67
phone,3,2.00
company,0,0.00
city,0,0.00
country,0,0.00
signup_date,0,0.00
status,0,0.00


### 5.Inspect duplicates

In [7]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 0


In [8]:
print("Duplicate customer IDs:", df["customer_id"].duplicated().sum())

Duplicate customer IDs: 0


### 6.Inspect all unique values in important columns

In [9]:
columns_to_inspect = ["city", "country", "status"]

for column in columns_to_inspect:
    print(f"\n--- {column.upper()} ---")
    print(df[column].unique())


--- CITY ---
<StringArray>
[   'New York', 'Los Angeles',     'Chicago',     'Seattle',      'Austin',
      'Boston',      'Denver',       'Miami',    'Portland',   'Bengaluru',
      'Mumbai',       'Delhi',   'Hyderabad',     'Chennai',        'Pune',
      'Jaipur',      'London',  'Manchester',     'Bristol']
Length: 19, dtype: str

--- COUNTRY ---
<StringArray>
['USA', 'India', 'UK', 'United Kingdom']
Length: 4, dtype: str

--- STATUS ---
<StringArray>
['Active', 'active', 'ACTIVE', 'Inactive', 'Pending', 'Unknown']
Length: 6, dtype: str


### 7.Detect leading/trailing whitespace

In [10]:
text_columns = df.select_dtypes(include="string").columns

whitespace_report = {}

for column in text_columns:
    mask = df[column].notna() & (
        df[column] != df[column].str.strip()
    )

    whitespace_report[column] = mask.sum()

whitespace_report

{'customer_id': np.int64(0),
 'first_name': np.int64(5),
 'last_name': np.int64(1),
 'email': np.int64(0),
 'phone': np.int64(0),
 'company': np.int64(0),
 'city': np.int64(0),
 'country': np.int64(0),
 'signup_date': np.int64(0),
 'status': np.int64(0)}

### 8.email validation

In [11]:
import re

email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

email_check = df.copy()

email_check["email_valid"] = email_check["email"].apply(
    lambda email: bool(re.match(email_pattern, email.strip()))
    if pd.notna(email) and email.strip() != ""
    else False
)

invalid_emails = email_check[
    email_check["email_valid"] == False
]

invalid_emails[["customer_id", "first_name", "last_name", "email"]]

,customer_id,first_name,last_name,email
5,C0006,Jessica,Miller,jessica.miller@example
47,C0048,Michelle,Collins,not-an-email
75,C0076,,Martin,NaN


### 9.find duplicate emails 

In [12]:
df["email_normalized"] = (
    df["email"]
    .str.strip()
    .str.lower()
)

duplicate_email_records = df[
    df["email_normalized"].notna()
    & df["email_normalized"].duplicated(keep=False)
].sort_values("email_normalized")

duplicate_email_records[
    ["customer_id", "first_name", "last_name", "email", "email_normalized"]
]

,customer_id,first_name,last_name,email,email_normalized
103,C0104,Ananya,Reddy,ananya.reddy@example.com,ananya.reddy@example.com
128,C0129,Ananya,Reddy,ananya.reddy@example.com,ananya.reddy@example.com
100,C0101,Arjun,Sharma,arjun.sharma@example.com,arjun.sharma@example.com
120,C0121,Arjun,Sharma,arjun.sharma@example.com,arjun.sharma@example.com
67,C0068,Jessica,Wood,jessica.wood@example.com,jessica.wood@example.com
87,C0088,Jessica,Wood,jessica.wood@example.com,jessica.wood@example.com
41,C0042,Jonathan,Phillips,jonathan.phillips@example.com,jonathan.phillips@example.com
96,C0097,Jonathan,Phillips,jonathan.phillips@example.com,jonathan.phillips@example.com
42,C0043,Raymond,Campbell,raymond.campbell@example.com,raymond.campbell@example.com
99,C0100,Raymond,Campbell,raymond.campbell@example.com,raymond.campbell@example.com


### 10.Inspect phone formats

In [13]:
phone_inspection = df[["customer_id", "phone"]].copy()

phone_inspection["phone_digits"] = (
    phone_inspection["phone"]
    .str.replace(r"\D", "", regex=True)
)

phone_inspection["phone_length"] = (
    phone_inspection["phone_digits"].str.len()
)

phone_inspection.head(20)

,customer_id,phone,phone_digits,phone_length
0,C0001,-1070,1070,4.0
1,C0002,4155550102,4155550102,10.0
2,C0003,+1 415 555 0103,14155550103,11.0
3,C0004,NaN,NaN,NaN
4,C0005,555-0105,5550105,7.0
5,C0006,415-555-0106,4155550106,10.0
6,C0007,-1076,1076,4.0
7,C0008,415.555.0108,4155550108,10.0
8,C0009,4155550109,4155550109,10.0
9,C0010,+1 415 555 0110,14155550110,11.0


### 10.Investigate unusual phone lengths

In [14]:
phone_inspection["phone_length"].value_counts(dropna=False)

phone_length
10.0    131
12.0      7
6.0       4
NaN       3
4.0       2
11.0      2
7.0       1
Name: count, dtype: int64

### 12.Inspect the raw date values

In [15]:
df["signup_date"].value_counts().head(30)

signup_date
01-03-2024    3
01/22/2024    2
20-03-2024    2
01-04-2024    2
15-04-2024    2
01-05-2024    2
01-06-2024    2
01-07-2024    2
15-07-2024    2
01-08-2024    2
15-08-2024    2
01-09-2024    2
10-09-2024    2
01-10-2024    2
01-11-2024    2
15-11-2024    2
01-12-2024    2
10-12-2024    2
01-02-2025    2
15-02-2025    2
01-03-2025    2
10-03-2025    2
10-04-2025    2
10-05-2025    2
10-06-2025    2
10-07-2025    2
10-08-2025    2
10-11-2025    2
05-01-2024    2
15-01-2024    1
Name: count, dtype: int64

### 13.Attempt date parsing

In [16]:
parsed_dates = pd.to_datetime(
    df["signup_date"],
    errors="coerce",
    format="mixed"
)

date_check = pd.DataFrame({
    "original_date": df["signup_date"],
    "parsed_date": parsed_dates
})

date_check.head(30)

,original_date,parsed_date
0,15-01-2024,2024-01-15
1,01/22/2024,2024-01-22
2,03-02-2024,2024-03-02
3,12-02-2024,2024-12-02
4,01-03-2024,2024-01-03
5,15-03-2024,2024-03-15
6,20-03-2024,2024-03-20
7,01-04-2024,2024-01-04
8,08-04-2024,2024-08-04
9,15-04-2024,2024-04-15


### 14.Identify dates that failed parsing

In [17]:
invalid_dates = date_check[
    date_check["parsed_date"].isna()
]

invalid_dates

,original_date,parsed_date


### 15.Inspect customer status

In [18]:
print("Unique statuses:")
print(df["status"].value_counts(dropna=False))

Unique statuses:
status
Active      75
Inactive    37
Pending     27
active       8
ACTIVE       2
Unknown      1
Name: count, dtype: int64


### 16.Find possible duplicate customers

In [19]:
df["name_key"] = (
    df["first_name"].fillna("").str.strip().str.lower()
    + " "
    + df["last_name"].fillna("").str.strip().str.lower()
)

potential_duplicate_names = df[
    df["name_key"].duplicated(keep=False)
].sort_values("name_key")

potential_duplicate_names[
    ["customer_id", "first_name", "last_name", "email", "phone", "company"]
]

,customer_id,first_name,last_name,email,phone,company
128,C0129,Ananya,Reddy,ananya.reddy@example.com,9876543029,Bright Solutions
103,C0104,Ananya,Reddy,ananya.reddy@example.com,NaN,Bright Solutions
100,C0101,Arjun,Sharma,arjun.sharma@example.com,-141675,Innovate Labs
120,C0121,Arjun,Sharma,arjun.sharma@example.com,-141675,Innovate Labs
67,C0068,Jessica,Wood,jessica.wood@example.com,4155550168,GreenLeaf
87,C0088,Jessica,Wood,jessica.wood@example.com,4155550168,GreenLeaf
41,C0042,Jonathan,Phillips,jonathan.phillips@example.com,4155550142,NorthStar
96,C0097,Jonathan,Phillips,jonathan.phillips@example.com,4155550197,CloudNine
47,C0048,Michelle,Collins,not-an-email,4155550148,BlueSky Ltd
97,C0098,Michelle,Collins,michelle.collins@example.com,4155550198,NorthStar
